# 10 — StationXML, network history and timing

Review the complete historical metadata separately from the waveform sample. This notebook calls the existing **10_conversion/network_history** reconstruction and writes a reproducible copy into this analysis run. It uses the current Matplotlib maps; ASL and FlowLoc are not involved.

The primary StationXML is provisional and has **no end-to-end counts response**. The separate nominal sensor/attenuation models output volts and must not be applied to digital counts as complete calibrations. Historical coordinates, elevations, gain boundaries and clock domains retain their qualifications.

Generated products use the shared `project_config.json` results root. Source/input files remain separate.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start in the Pinatubo repository; project_config.json must be available.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pinatubo_project import input_path, output_path, runtime_environment
runtime_environment()

from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))


In [ ]:
from analysis_workflow import review_network
REBUILD_FROM_EVIDENCE = True
channels, inventory_xml = review_network(ctx, rebuild=REBUILD_FROM_EVIDENCE)
NETWORK = ctx['out'] / 'network_history'
print(inventory_xml)
display(channels)
display(pd.read_csv(NETWORK / 'pinatubo_channel_epochs.csv'))
display(pd.read_csv(NETWORK / 'alternative_coordinates.csv'))


## Maps, telemetry and operations

Early June represents the network before the documented June 11 link losses. July/August is a union of observed header identities; it does not prove continuous operation or installation dates. Modern topography is not the 1991 surface. Unverified PVO/repeater coordinates are not invented.

In [ ]:
for name in ['network_early_june', 'network_july_august', 'telemetry', 'pvo_operations']:
    display(Image(filename=str(NETWORK / f'{name}.png')))


In [ ]:
display(Markdown((NETWORK / 'chronology.md').read_text()))
display(Markdown((NETWORK / 'timing_issues.md').read_text()))


## Response-model audit

Review stage units and nominal sensitivities before doing any response correction. Unknown electronics remain explicit. These models do not justify physical cross-station amplitude comparisons. Matching a recorded timestamp to an approximate gain epoch is also provisional when clocks are uncertain.

In [ ]:
from obspy import read_inventory
nominal = read_inventory(str(NETWORK / 'pinatubo_nominal_voltage_models.xml'))
rows = []
for network in nominal:
    for station in network:
        for channel in station:
            response = channel.response
            if response is None:
                continue
            for stage in response.response_stages:
                rows.append(dict(station=station.code, channel=channel.code,
                    epoch_start=str(channel.start_date), stage=stage.stage_sequence_number,
                    input_units=stage.input_units, output_units=stage.output_units,
                    stage_gain=stage.stage_gain, gain_frequency_hz=stage.stage_gain_frequency,
                    status='NOMINAL_MODEL; UNKNOWN_ELECTRONICS'))
display(pd.DataFrame(rows))
print('Analysis metrics retain counts; no response removal is performed.')


Review the source JSON/evidence under `10_conversion/network_history/data` when metadata need changing; rerun this notebook afterwards. Network source hashes are saved with these products. Continue to notebook **20**.